In [1]:
import time
import random
import logging
from typing import Optional
from io import StringIO

import requests
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/121.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch_with_retry(
    url: str,
    max_retries: int = 5,
    base_backoff: float = 1.0,
    timeout: float = 12.0
) -> Optional[str]:
    """
    Fetch a URL with retries + exponential backoff.
    Returns HTML text or None if all retries fail.
    """
    for attempt in range(1, max_retries + 1):
        try:
            logging.info(f"GET {url} (attempt {attempt}/{max_retries})")
            resp = requests.get(url, headers=HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return resp.text
            else:
                logging.warning(f"Status {resp.status_code} for {url}")
        except Exception as e:
            logging.warning(f"Error fetching {url}: {e}")

        sleep_for = base_backoff * (2 ** (attempt - 1)) + random.uniform(0, 1.0)
        logging.info(f"Sleeping {sleep_for:.1f}s before retry")
        time.sleep(sleep_for)

    logging.error(f"Failed after {max_retries} retries: {url}")
    return None


In [3]:
def get_games_for_season(season: int) -> pd.DataFrame:
    """
    Scrape all NHL games for a given season (ending year, e.g. 2024 for 2023–24)
    from the 'Schedule and Results' page.
    Returns one row per game with home/away teams, scores, and outcome.
    """
    url = f"https://www.hockey-reference.com/leagues/NHL_{season}_games.html"
    html = fetch_with_retry(url)
    if html is None:
        raise RuntimeError(f"Could not fetch schedule page for {season}")

    # Use StringIO to avoid FutureWarning from pandas
    all_tables = pd.read_html(StringIO(html))
    game_tables = []

    for df in all_tables:
        cols_lower = [str(c).lower() for c in df.columns]
        if "date" in cols_lower and ("visitor" in cols_lower or "away" in cols_lower):
            game_tables.append(df)

    if not game_tables:
        raise RuntimeError(f"No game tables found for season {season}")

    games = pd.concat(game_tables, ignore_index=True)
    games.columns = [str(c).strip() for c in games.columns]

    # Drop header / "Playoffs" rows
    games = games[~games["Date"].astype(str).str.contains("Playoffs|Date", na=False)]

    # Drop rows with missing team names
    games = games.dropna(subset=["Date", "Visitor", "Home"], how="any")

    # basic fields
    games["season"] = season
    games["game_date"] = pd.to_datetime(games["Date"])

    # Map to away/home teams + goals
    col_map = {}
    for col in games.columns:
        lc = col.lower()
        if lc == "visitor":
            col_map[col] = "away_team"
        elif lc == "home":
            col_map[col] = "home_team"
        elif lc == "g":
            if "away_goals" not in col_map.values():
                col_map[col] = "away_goals"
        elif lc == "g.1":
            col_map[col] = "home_goals"

    games = games.rename(columns=col_map)

    # Drop rows without scores and cast to int
    games = games.dropna(subset=["away_goals", "home_goals"])
    games["away_goals"] = games["away_goals"].astype(int)
    games["home_goals"] = games["home_goals"].astype(int)

    # Outcomes
    games["home_win"] = (games["home_goals"] > games["away_goals"]).astype(int)
    games["home_win_margin"] = games["home_goals"] - games["away_goals"]

    # Simple game_id
    games["game_id"] = (
        games["game_date"].dt.strftime("%Y%m%d")
        + "_"
        + games["away_team"].str.replace(" ", "", regex=False)
        + "_@_"
        + games["home_team"].str.replace(" ", "", regex=False)
    )

    keep_cols = [
        "game_id", "season", "game_date",
        "away_team", "away_goals",
        "home_team", "home_goals",
        "home_win", "home_win_margin"
    ]

    return games[keep_cols].reset_index(drop=True)


In [5]:
start_season = 2010  
end_season = 2024

all_games = []

for season in range(start_season, end_season + 1):
    print(f"\n=== Scraping {season} ===")
    g = get_games_for_season(season)
    all_games.append(g)
    print(f"{season}: {len(g)} games scraped")

    # polite delay
    time.sleep(random.uniform(1.0, 2.5))

games_df = pd.concat(all_games, ignore_index=True)
print("All games shape:", games_df.shape)

games_df.head()


2025-11-14 11:28:07,775 | INFO | GET https://www.hockey-reference.com/leagues/NHL_2010_games.html (attempt 1/5)



=== Scraping 2010 ===
2010: 1319 games scraped


2025-11-14 11:28:09,669 | INFO | GET https://www.hockey-reference.com/leagues/NHL_2011_games.html (attempt 1/5)



=== Scraping 2011 ===
2011: 1319 games scraped


2025-11-14 11:28:12,733 | INFO | GET https://www.hockey-reference.com/leagues/NHL_2012_games.html (attempt 1/5)


...

2025-11-14 11:28:35,873 | INFO | GET https://www.hockey-reference.com/leagues/NHL_2023_games.html (attempt 1/5)



=== Scraping 2023 ===
2023: 1400 games scraped


2025-11-14 11:28:37,806 | INFO | GET https://www.hockey-reference.com/leagues/NHL_2024_games.html (attempt 1/5)



=== Scraping 2024 ===
2024: 1400 games scraped
All games shape: (19118, 9)


,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin
0,20091001_WashingtonCapitals_@_BostonBruins,2010,2009-10-01,Washington Capitals,4,Boston Bruins,1,0,-3
1,20091001_VancouverCanucks_@_CalgaryFlames,2010,2009-10-01,Vancouver Canucks,3,Calgary Flames,5,1,2
2,20091001_SanJoseSharks_@_ColoradoAvalanche,2010,2009-10-01,San Jose Sharks,2,Colorado Avalanche,5,1,3
3,20091001_MontrealCanadiens_@_TorontoMapleLeafs,2010,2009-10-01,Montreal Canadiens,4,Toronto Maple Leafs,3,0,-1
4,20091002_PhiladelphiaFlyers_@_CarolinaHurricanes,2010,2009-10-02,Philadelphia Flyers,2,Carolina Hurricanes,0,0,-2


In [7]:
# Long format: one row per team-game (so we can group by team)
home = games_df.rename(
    columns={
        "home_team": "team",
        "home_goals": "goals_for",
        "away_goals": "goals_against"
    }
)[["season", "game_date", "team", "goals_for", "goals_against"]]

away = games_df.rename(
    columns={
        "away_team": "team",
        "away_goals": "goals_for",
        "home_goals": "goals_against"
    }
)[["season", "game_date", "team", "goals_for", "goals_against"]]

team_games = pd.concat([home, away], ignore_index=True)

# Team-season aggregates: GF, GA, GF/G, GA/G, games played
team_season_stats = (
    team_games
    .groupby(["season", "team"], as_index=False)
    .agg(
        gp=("game_date", "count"),
        gf=("goals_for", "sum"),
        ga=("goals_against", "sum")
    )
)

team_season_stats["gf_per_game"] = team_season_stats["gf"] / team_season_stats["gp"]
team_season_stats["ga_per_game"] = team_season_stats["ga"] / team_season_stats["gp"]

team_season_stats.head()


,season,team,gp,gf,ga,gf_per_game,ga_per_game
0,2010,Anaheim Ducks,82,238,251,2.902439,3.060976
1,2010,Atlanta Thrashers,82,234,256,2.853659,3.121951
2,2010,Boston Bruins,95,242,237,2.547368,2.494737
3,2010,Buffalo Sabres,88,250,223,2.840909,2.534091
4,2010,Calgary Flames,82,204,210,2.487805,2.560976


In [9]:
games_df.to_csv(f"nhl_games_{start_season}_{end_season}.csv", index=False)
team_season_stats.to_csv(f"nhl_team_season_stats_{start_season}_{end_season}.csv", index=False)

Get game stats

In [14]:
import time
import random
import logging
from io import StringIO
import requests
import pandas as pd

logging.basicConfig(level=logging.INFO)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/121.0.0.0 Safari/537.36"
    )
}

def fetch_with_retry(url, max_retries=5):
    for attempt in range(1, max_retries+1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            if resp.status_code == 200:
                return resp.text
        except Exception as e:
            logging.warning(f"Attempt {attempt} failed: {e}")
        time.sleep(random.uniform(1, 3))
    raise RuntimeError(f"Failed to fetch: {url}")

def get_games_for_season(season):
    url = f"https://www.hockey-reference.com/leagues/NHL_{season}_games.html"
    html = fetch_with_retry(url)

    tables = pd.read_html(StringIO(html))

    # Identify schedule tables
    game_tables = []
    for df in tables:
        cols = [str(c).lower() for c in df.columns]
        if "date" in cols and ("visitor" in cols or "away" in cols):
            game_tables.append(df)

    if not game_tables:
        raise RuntimeError(f"No game tables found for season {season}")

    games = pd.concat(game_tables, ignore_index=True)
    games.columns = [str(c).strip() for c in games.columns]

    # Drop rows like "Playoffs"
    games = games[~games["Date"].astype(str).str.contains("Playoffs|Date", na=False)]
    games = games.dropna(subset=["Date", "Visitor", "Home"])

    # Rename
    rename_map = {}
    for col in games.columns:
        lc = col.lower()
        if lc == "visitor": rename_map[col] = "away_team"
        if lc == "home": rename_map[col] = "home_team"
        if lc == "g": rename_map[col] = "away_goals"
        if lc == "g.1": rename_map[col] = "home_goals"

    games = games.rename(columns=rename_map)

    games = games.dropna(subset=["away_goals", "home_goals"])
    games["away_goals"] = games["away_goals"].astype(int)
    games["home_goals"] = games["home_goals"].astype(int)

    games["season"] = season
    games["game_date"] = pd.to_datetime(games["Date"])

    games["home_win"] = (games["home_goals"] > games["away_goals"]).astype(int)
    games["home_win_margin"] = games["home_goals"] - games["away_goals"]

    games["game_id"] = (
        games["game_date"].dt.strftime("%Y%m%d") + "_" +
        games["away_team"].str.replace(" ", "", regex=False) +
        "_@_" +
        games["home_team"].str.replace(" ", "", regex=False)
    )

    return games[[
        "game_id", "season", "game_date",
        "away_team", "away_goals",
        "home_team", "home_goals",
        "home_win", "home_win_margin"
    ]]


In [16]:
start_season = 2010
end_season = 2024

all_games = []

for season in range(start_season, end_season+1):
    print(f"Scraping {season}...")
    df = get_games_for_season(season)
    all_games.append(df)
    time.sleep(random.uniform(1.0, 2.0))

games_df = pd.concat(all_games, ignore_index=True)
games_df.shape


Scraping 2010...
Scraping 2011...
Scraping 2012...
Scraping 2013...
Scraping 2014...
Scraping 2015...
Scraping 2016...
Scraping 2017...
Scraping 2018...
Scraping 2019...
Scraping 2020...
Scraping 2021...
Scraping 2022...
Scraping 2023...
Scraping 2024...


(19118, 9)

In [20]:
games_df.to_csv("nhl_game_logs_2010_2024.csv", index=False)
games_df.head()


,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin
0,20091001_WashingtonCapitals_@_BostonBruins,2010,2009-10-01,Washington Capitals,4,Boston Bruins,1,0,-3
1,20091001_VancouverCanucks_@_CalgaryFlames,2010,2009-10-01,Vancouver Canucks,3,Calgary Flames,5,1,2
2,20091001_SanJoseSharks_@_ColoradoAvalanche,2010,2009-10-01,San Jose Sharks,2,Colorado Avalanche,5,1,3
3,20091001_MontrealCanadiens_@_TorontoMapleLeafs,2010,2009-10-01,Montreal Canadiens,4,Toronto Maple Leafs,3,0,-1
4,20091002_PhiladelphiaFlyers_@_CarolinaHurricanes,2010,2009-10-02,Philadelphia Flyers,2,Carolina Hurricanes,0,0,-2


Scrape advanced stats

In [51]:
import time
import logging
import random
from typing import List, Optional
from io import StringIO

import requests
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/121.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})


def fetch_with_retries(
    url: str,
    max_attempts: int = 3,
    backoff_factor: float = 2.0,
    timeout: int = 20,
) -> Optional[str]:
    """
    Polite GET with exponential backoff.
    Returns HTML text on success, or None on failure.
    """
    for attempt in range(1, max_attempts + 1):
        try:
            logging.info("GET %s (attempt %d/%d)", url, attempt, max_attempts)
            resp = SESSION.get(url, timeout=timeout)
            if resp.status_code == 200:
                return resp.text

            # retryable server issues
            if resp.status_code in (429, 500, 502, 503, 504):
                sleep_s = backoff_factor ** attempt
                logging.warning(
                    "HTTP %s for %s. Sleeping %.1fs before retry.",
                    resp.status_code, url, sleep_s
                )
                time.sleep(sleep_s)
                continue

            logging.error("Non-OK status %s for %s", resp.status_code, url)
            return None

        except (requests.ConnectionError, requests.Timeout) as e:
            sleep_s = backoff_factor ** attempt
            logging.warning("Request error %r. Sleeping %.1fs before retry.", e, sleep_s)
            time.sleep(sleep_s)

    logging.error("Failed to fetch %s after %d attempts", url, max_attempts)
    return None


In [53]:
# Modern NHL team codes; tweak if you want a subset
TEAM_CODES = [
    "ANA", "ARI", "BOS", "BUF", "CAR", "CBJ", "CGY", "CHI",
    "COL", "DAL", "DET", "EDM", "FLA", "LAK", "MIN", "MTL",
    "NJD", "NSH", "NYI", "NYR", "OTT", "PHI", "PIT", "SEA",
    "SJS", "STL", "TBL", "TOR", "VAN", "VGK", "WPG", "WSH"
    # add "UTA" later if NST adopts it and you need it
]

def season_to_nst(start_year: int) -> str:
    """
    Convert a start year like 2024 -> '20242025'
    which is what NST expects in ?season=...
    """
    return f"{start_year}{start_year + 1}"


def nst_team_code(team: str, start_year: int) -> str:
    """
    Handle historical name changes if needed.
    Example: ARI used to be PHX before 2014-15.
    """
    if team == "ARI" and start_year < 2014:
        return "PHX"
    return team


In [55]:
def scrape_team_season_report(start_year: int, team: str) -> Optional[pd.DataFrame]:
    """
    Scrape a single team-season report from Natural Stat Trick.

    start_year: e.g., 2024 for the 2024-25 season.
    team: 3-letter team code (e.g., 'BOS').

    Returns a DataFrame with Season + Team columns, or None if failed.
    """
    season_str = season_to_nst(start_year)
    nst_team = nst_team_code(team, start_year)

    url = (
        "https://www.naturalstattrick.com/teamreport.php"
        f"?season={season_str}&team={nst_team}&stype=2"
    )
    html = fetch_with_retries(url)

    if html is None:
        logging.error("Skipping %s %s (could not fetch page).", season_str, team)
        return None

    try:
        tables = pd.read_html(StringIO(html), header=0)
    except ValueError as e:
        logging.error("No tables found for %s %s: %r", season_str, team, e)
        return None

    if not tables:
        logging.warning("No tables parsed for %s %s", season_str, team)
        return None

    df = tables[0].copy()

    # annotate + clean columns
    df["Season"] = f"{start_year}-{start_year + 1}"
    df["Team"] = team
    df.columns = [str(c).strip().replace(" ", "_").replace("%", "Pct") for c in df.columns]

    return df


In [57]:
def scrape_nst_season(
    start_season: int,
    teams: Optional[List[str]] = None,
    min_sleep: float = 6.0,
    max_sleep: float = 12.0,
) -> pd.DataFrame:
    """
    Scrape Natural Stat Trick team season totals for ONE NHL season.

    start_season: e.g., 2024 for the 2024-25 season.
    teams: list of team codes to scrape. Default = all TEAM_CODES.
    min_sleep, max_sleep: random sleep range between team requests, in seconds.

    Returns a concatenated DataFrame of all successfully scraped teams.
    """
    if teams is None:
        teams = TEAM_CODES

    season_label = f"{start_season}-{start_season + 1}"
    logging.info("=== Scraping NST season %s ===", season_label)

    all_team_rows = []

    for team in teams:
        logging.info("Scraping season %s team %s", season_label, team)
        try:
            df_team = scrape_team_season_report(start_season, team)
            if df_team is not None and not df_team.empty:
                all_team_rows.append(df_team)
        except Exception as e:
            logging.error(
                "Unexpected failure for season %s team %s: %r",
                season_label, team, e
            )

        # VERY polite random sleep between requests
        sleep_s = random.uniform(min_sleep, max_sleep)
        logging.info("Sleeping %.1fs before next team...", sleep_s)
        time.sleep(sleep_s)

    if not all_team_rows:
        raise RuntimeError(f"No NST data scraped for season {season_label}; check connectivity or parameters.")

    full_df = pd.concat(all_team_rows, ignore_index=True)
    logging.info(
        "Finished season %s: scraped %d rows for %d teams.",
        season_label, len(full_df), full_df['Team'].nunique()
    )
    return full_df


In [59]:
nst_2024 = scrape_nst_season(start_season=2024)
nst_2024.to_csv("nst_team_season_2024_2025.csv", index=False)
nst_2024.head()


2025-11-14 12:21:07,300 | INFO | === Scraping NST season 2024-2025 ===
2025-11-14 12:21:07,301 | INFO | Scraping season 2024-2025 team ANA
2025-11-14 12:21:07,301 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20242025&team=ANA&stype=2 (attempt 1/3)
2025-11-14 12:21:08,049 | INFO | Sleeping 9.2s before next team...
2025-11-14 12:21:17,220 | INFO | Scraping season 2024-2025 team ARI
2025-11-14 12:21:17,221 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20242025&team=ARI&stype=2 (attempt 1/3)
2025-11-14 12:21:17,662 | ERROR | Unexpected failure for season 2024-2025 team ARI: ImportError("Missing optional dependency 'html5lib'.  Use pip or conda to install html5lib.")
2025-11-14 12:21:17,663 | INFO | Sleeping 8.2s before next team...
2025-11-14 12:21:25,899 | INFO | Scraping season 2024-2025 team BOS
2025-11-14 12:21:25,901 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20242025&team=BOS&stype=2 (attempt 1/3)
2025-11-14 12:21:

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,82,4989.2667,4754.00,5549.00,46.14,3452.00,4040.00,46.08,2270.00,...,107.0,144.00,42.63,19.38,79.55,9.56,90.15,0.997,2024-2025,ANA
1,Even Strength,82,4192.2667,4011.00,4748.00,45.79,2883.00,3414.00,45.78,1891.00,...,93.0,108.00,46.27,20.67,80.95,9.89,91.38,1.013,2024-2025,ANA
2,5v5,82,3969.3667,3772.00,4445.00,45.90,2709.00,3194.00,45.89,1772.00,...,84.0,97.00,46.41,20.14,81.84,8.80,92.05,1.009,2024-2025,ANA
3,5v5 Score & Venue Adjusted,82,3969.3667,3690.32,4473.07,45.21,2660.19,3211.69,45.30,1741.48,...,83.3,98.05,45.93,20.45,81.72,8.83,92.04,1.009,2024-2025,ANA
4,5v4 Power Play,80,376.6500,596.00,80.00,88.17,452.00,73.00,86.10,290.00,...,9.0,2.00,81.82,12.68,80.00,7.93,90.91,0.988,2024-2025,ANA


In [63]:
nst_2023 = scrape_nst_season(start_season=2023)
nst_2023.to_csv("nst_team_season_2023_2024.csv", index=False)
nst_2023.head()

2025-11-14 12:42:48,265 | INFO | === Scraping NST season 2023-2024 ===
2025-11-14 12:42:48,267 | INFO | Scraping season 2023-2024 team ANA
2025-11-14 12:42:48,268 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20232024&team=ANA&stype=2 (attempt 1/3)
2025-11-14 12:42:48,954 | INFO | Sleeping 10.4s before next team...
2025-11-14 12:42:59,333 | INFO | Scraping season 2023-2024 team ARI
2025-11-14 12:42:59,334 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20232024&team=ARI&stype=2 (attempt 1/3)
2025-11-14 12:42:59,998 | INFO | Sleeping 7.2s before next team...
2025-11-14 12:43:07,187 | INFO | Scraping season 2023-2024 team BOS
2025-11-14 12:43:07,189 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20232024&team=BOS&stype=2 (attempt 1/3)
2025-11-14 12:43:07,868 | INFO | Sleeping 11.3s before next team...
2025-11-14 12:43:19,158 | INFO | Scraping season 2023-2024 team BUF
2025-11-14 12:43:19,161 | INFO | GET https://www.naturalst

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,82,4951.6500,4331.00,5309.00,44.93,3156.00,3857.00,45.00,2194.00,...,102.00,142.00,41.80,19.10,80.94,9.25,89.01,0.983,2023-2024,ANA
1,Even Strength,82,4047.2000,3584.00,4126.00,46.49,2601.00,2985.00,46.56,1787.00,...,81.00,100.00,44.75,18.45,81.85,8.34,90.37,0.987,2023-2024,ANA
2,5v5,82,3822.6833,3344.00,3864.00,46.39,2423.00,2790.00,46.48,1667.00,...,73.00,93.00,43.98,18.02,81.94,7.62,90.94,0.986,2023-2024,ANA
3,5v5 Score & Venue Adjusted,82,3822.6833,3235.22,3943.08,45.07,2359.07,2840.87,45.37,1623.33,...,72.73,94.46,43.50,18.35,82.00,7.76,91.01,0.988,2023-2024,ANA
4,5v4 Power Play,81,366.2167,568.00,77.00,88.06,404.00,64.00,86.32,289.00,...,13.00,2.00,86.67,18.57,89.47,13.15,92.45,1.056,2023-2024,ANA


In [64]:
nst_2022 = scrape_nst_season(start_season=2022)
nst_2022.to_csv("nst_team_season_2022_2023.csv", index=False)
nst_2022.head()

2025-11-14 12:47:49,194 | INFO | === Scraping NST season 2022-2023 ===
2025-11-14 12:47:49,195 | INFO | Scraping season 2022-2023 team ANA
2025-11-14 12:47:49,195 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20222023&team=ANA&stype=2 (attempt 1/3)
2025-11-14 12:47:49,871 | INFO | Sleeping 8.7s before next team...
2025-11-14 12:47:58,622 | INFO | Scraping season 2022-2023 team ARI
2025-11-14 12:47:58,624 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20222023&team=ARI&stype=2 (attempt 1/3)
2025-11-14 12:47:59,315 | INFO | Sleeping 10.6s before next team...
2025-11-14 12:48:09,881 | INFO | Scraping season 2022-2023 team BOS
2025-11-14 12:48:09,883 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20222023&team=BOS&stype=2 (attempt 1/3)
2025-11-14 12:48:10,558 | INFO | Sleeping 10.5s before next team...
2025-11-14 12:48:21,038 | INFO | Scraping season 2022-2023 team BUF
2025-11-14 12:48:21,040 | INFO | GET https://www.naturalst

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,82,4985.6333,4234.00,5565.00,43.21,3184.00,4364.00,42.18,2326.00,...,102.00,175.00,36.82,16.37,82.57,8.86,89.56,0.984,2022-2023,ANA
1,Even Strength,82,4172.9167,3477.00,4638.00,42.85,2603.00,3605.00,41.93,1907.00,...,85.00,138.00,38.12,16.31,83.45,8.86,90.58,0.994,2022-2023,ANA
2,5v5,82,3992.5000,3300.00,4433.00,42.67,2464.00,3444.00,41.71,1801.00,...,73.00,129.00,36.14,15.08,83.59,7.94,91.17,0.991,2022-2023,ANA
3,5v5 Score & Venue Adjusted,82,3992.5000,3153.61,4580.49,40.78,2371.95,3547.11,40.07,1739.14,...,72.18,130.27,35.65,15.43,83.90,8.12,91.33,0.994,2022-2023,ANA
4,5v4 Power Play,79,362.2333,630.00,89.00,87.62,474.00,80.00,85.56,331.00,...,16.00,4.00,80.00,18.18,82.61,10.57,90.91,1.015,2022-2023,ANA


In [66]:
nst_2021 = scrape_nst_season(start_season=2021)
nst_2021.to_csv("nst_team_season_2021_2022.csv", index=False)
nst_2021.head()

2025-11-14 12:53:10,233 | INFO | === Scraping NST season 2021-2022 ===
2025-11-14 12:53:10,234 | INFO | Scraping season 2021-2022 team ANA
2025-11-14 12:53:10,234 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20212022&team=ANA&stype=2 (attempt 1/3)
2025-11-14 12:53:10,923 | INFO | Sleeping 7.8s before next team...
2025-11-14 12:53:18,683 | INFO | Scraping season 2021-2022 team ARI
2025-11-14 12:53:18,686 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20212022&team=ARI&stype=2 (attempt 1/3)
2025-11-14 12:53:19,479 | INFO | Sleeping 7.4s before next team...
2025-11-14 12:53:26,891 | INFO | Scraping season 2021-2022 team BOS
2025-11-14 12:53:26,893 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20212022&team=BOS&stype=2 (attempt 1/3)
2025-11-14 12:53:27,570 | INFO | Sleeping 6.8s before next team...
2025-11-14 12:53:34,363 | INFO | Scraping season 2021-2022 team BUF
2025-11-14 12:53:34,365 | INFO | GET https://www.naturalstat

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,82,5001.9833,4399.00,4893.00,47.34,3326.00,3771.00,46.86,2395.00,...,127.00,132.00,49.03,18.98,81.99,9.52,90.23,0.997,2021-2022,ANA
1,Even Strength,82,4251.3167,3744.00,4126.00,47.57,2846.00,3139.00,47.55,2043.00,...,98.00,111.00,46.89,16.90,81.83,8.42,90.29,0.987,2021-2022,ANA
2,5v5,82,4035.9667,3551.00,3877.00,47.81,2699.00,2942.00,47.85,1937.00,...,92.00,95.00,49.20,16.61,83.30,7.95,91.14,0.991,2021-2022,ANA
3,5v5 Score & Venue Adjusted,82,4035.9667,3506.94,3888.17,47.42,2670.41,2955.15,47.47,1918.48,...,91.79,94.28,49.33,16.79,83.56,8.01,91.18,0.992,2021-2022,ANA
4,5v4 Power Play,82,360.1000,555.00,93.00,85.65,390.00,84.00,82.28,279.00,...,23.00,3.00,88.46,32.39,82.35,16.49,95.65,1.121,2021-2022,ANA


In [68]:
nst_2020 = scrape_nst_season(start_season=2020)
nst_2020.to_csv("nst_team_season_2020_2021.csv", index=False)
nst_2020.head()

2025-11-14 12:58:16,820 | INFO | === Scraping NST season 2020-2021 ===
2025-11-14 12:58:16,820 | INFO | Scraping season 2020-2021 team ANA
2025-11-14 12:58:16,820 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20202021&team=ANA&stype=2 (attempt 1/3)
2025-11-14 12:58:17,495 | INFO | Sleeping 6.3s before next team...
2025-11-14 12:58:23,809 | INFO | Scraping season 2020-2021 team ARI
2025-11-14 12:58:23,810 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20202021&team=ARI&stype=2 (attempt 1/3)
2025-11-14 12:58:24,494 | INFO | Sleeping 7.1s before next team...
2025-11-14 12:58:31,577 | INFO | Scraping season 2020-2021 team BOS
2025-11-14 12:58:31,577 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20202021&team=BOS&stype=2 (attempt 1/3)
2025-11-14 12:58:32,178 | INFO | Sleeping 10.9s before next team...
2025-11-14 12:58:43,043 | INFO | Scraping season 2020-2021 team BUF
2025-11-14 12:58:43,045 | INFO | GET https://www.naturalsta

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,56,3400.1833,2910.00,3099.00,48.43,2157.00,2363.00,47.72,1500.00,...,69.00,105.00,39.66,17.51,79.08,8.27,89.67,0.979,2020-2021,ANA
1,Even Strength,56,2886.8667,2495.00,2608.00,48.89,1828.00,1977.00,48.04,1268.00,...,60.00,89.00,40.27,17.70,78.50,8.75,90.29,0.990,2020-2021,ANA
2,5v5,56,2764.6500,2379.00,2494.00,48.82,1740.00,1888.00,47.96,1211.00,...,57.00,80.00,41.61,17.76,79.43,8.26,91.06,0.993,2020-2021,ANA
3,5v5 Score & Venue Adjusted,56,2764.6500,2323.78,2525.25,47.92,1703.04,1910.47,47.13,1188.54,...,56.61,81.11,41.11,18.04,79.51,8.41,91.05,0.995,2020-2021,ANA
4,5v4 Power Play,55,209.3167,316.00,47.00,87.05,244.00,44.00,84.72,163.00,...,4.00,1.00,80.00,10.53,85.71,4.29,90.32,0.946,2020-2021,ANA


2019 didn't work...

In [72]:
nst_2019 = scrape_nst_season(start_season=2019)
nst_2019.to_csv("nst_team_season_2019_2020.csv", index=False)
nst_2019.head()

2025-11-14 13:12:33,227 | INFO | === Scraping NST season 2019-2020 ===
2025-11-14 13:12:33,228 | INFO | Scraping season 2019-2020 team ANA
2025-11-14 13:12:33,229 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20192020&team=ANA&stype=2 (attempt 1/3)
2025-11-14 13:12:34,201 | INFO | Sleeping 9.3s before next team...
2025-11-14 13:12:43,505 | INFO | Scraping season 2019-2020 team ARI
2025-11-14 13:12:43,507 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20192020&team=ARI&stype=2 (attempt 1/3)
2025-11-14 13:12:44,340 | INFO | Sleeping 11.1s before next team...
2025-11-14 13:12:55,463 | INFO | Scraping season 2019-2020 team BOS
2025-11-14 13:12:55,465 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20192020&team=BOS&stype=2 (attempt 1/3)
2025-11-14 13:12:56,270 | INFO | Sleeping 9.9s before next team...
2025-11-14 13:13:06,128 | INFO | Scraping season 2019-2020 team BUF
2025-11-14 13:13:06,129 | INFO | GET https://www.naturalsta

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,71,4320.1833,3916.00,4147.00,48.57,3011.00,3203.00,48.46,2108.00,...,99.00,113.00,46.70,17.43,82.09,8.63,90.15,0.988,2019-2020,ANA
1,Even Strength,71,3612.2667,3315.00,3564.00,48.19,2549.00,2737.00,48.22,1758.00,...,76.00,90.00,45.78,16.56,82.79,8.19,91.24,0.994,2019-2020,ANA
2,5v5,71,3430.1167,3144.00,3364.00,48.31,2414.00,2581.00,48.33,1675.00,...,68.00,86.00,44.16,15.70,82.45,7.76,91.83,0.996,2019-2020,ANA
3,5v5 Score & Venue Adjusted,71,3430.1167,3092.26,3365.54,47.88,2379.08,2581.59,47.96,1654.38,...,67.83,85.88,44.13,15.99,82.56,7.83,91.87,0.997,2019-2020,ANA
4,5v4 Power Play,71,312.7667,468.00,74.00,86.35,347.00,64.00,84.43,256.00,...,13.00,2.00,86.67,16.67,84.62,9.77,90.91,1.007,2019-2020,ANA


In [74]:
nst_2018 = scrape_nst_season(start_season=2018)
nst_2018.to_csv("nst_team_season_2018_2019.csv", index=False)
nst_2018.head()

2025-11-14 13:27:56,292 | INFO | === Scraping NST season 2018-2019 ===
2025-11-14 13:27:56,294 | INFO | Scraping season 2018-2019 team ANA
2025-11-14 13:27:56,295 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20182019&team=ANA&stype=2 (attempt 1/3)
2025-11-14 13:27:57,156 | INFO | Sleeping 9.3s before next team...
2025-11-14 13:28:06,508 | INFO | Scraping season 2018-2019 team ARI
2025-11-14 13:28:06,510 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20182019&team=ARI&stype=2 (attempt 1/3)
2025-11-14 13:28:07,382 | INFO | Sleeping 10.1s before next team...
2025-11-14 13:28:17,509 | INFO | Scraping season 2018-2019 team BOS
2025-11-14 13:28:17,511 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20182019&team=BOS&stype=2 (attempt 1/3)
2025-11-14 13:28:18,411 | INFO | Sleeping 7.9s before next team...
2025-11-14 13:28:26,336 | INFO | Scraping season 2018-2019 team BUF
2025-11-14 13:28:26,338 | INFO | GET https://www.naturalsta

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,82,4971.9833,4394.0,5051.0,46.52,3278.00,3855.00,45.96,2272.00,...,110.00,139.00,44.18,18.61,83.43,8.63,90.91,0.995,2018-2019,ANA
1,Even Strength,82,4147.0000,3744.0,4163.0,47.35,2783.00,3145.00,46.95,1925.00,...,85.00,107.00,44.27,17.21,83.69,8.00,91.65,0.996,2018-2019,ANA
2,5v5,82,3975.5667,3595.0,3957.0,47.60,2672.00,2984.00,47.24,1845.00,...,78.00,95.00,45.09,16.63,84.48,7.37,92.55,0.999,2018-2019,ANA
3,5v5 Score & Venue Adjusted,82,3975.5667,3515.9,3983.5,46.88,2625.05,3002.53,46.65,1817.81,...,76.98,96.02,44.50,16.71,84.47,7.42,92.60,1.000,2018-2019,ANA
4,5v4 Power Play,82,347.6000,537.0,105.0,83.64,392.00,93.00,80.82,280.00,...,19.00,4.00,82.61,23.46,84.00,12.86,86.96,0.998,2018-2019,ANA


In [90]:
nst_2010= scrape_nst_season(start_season=2010)
nst_2010.to_csv("nst_team_season_2010_2011.csv", index=False)
nst_2010.head()

2025-11-14 15:32:37,539 | INFO | === Scraping NST season 2010-2011 ===
2025-11-14 15:32:37,541 | INFO | Scraping season 2010-2011 team ANA
2025-11-14 15:32:37,542 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20102011&team=ANA&stype=2 (attempt 1/3)
2025-11-14 15:32:38,293 | INFO | Sleeping 6.9s before next team...
2025-11-14 15:32:45,164 | INFO | Scraping season 2010-2011 team ARI
2025-11-14 15:32:45,165 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20102011&team=PHX&stype=2 (attempt 1/3)
2025-11-14 15:32:45,903 | INFO | Sleeping 11.3s before next team...
2025-11-14 15:32:57,207 | INFO | Scraping season 2010-2011 team BOS
2025-11-14 15:32:57,208 | INFO | GET https://www.naturalstattrick.com/teamreport.php?season=20102011&team=BOS&stype=2 (attempt 1/3)
2025-11-14 15:32:57,929 | INFO | Sleeping 11.0s before next team...
2025-11-14 15:33:08,917 | INFO | Scraping season 2010-2011 team BUF
2025-11-14 15:33:08,919 | INFO | GET https://www.naturalst

,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,82,4992.7167,4093.00,5042.00,44.81,3219.00,3794.00,45.90,2334.00,...,143.00,131.00,52.19,20.52,82.56,10.07,91.20,1.013,2010-2011,ANA
1,Even Strength,82,4039.4167,3221.00,4010.00,44.54,2509.00,3038.00,45.23,1812.00,...,95.00,93.00,50.53,18.45,84.32,8.72,91.99,1.007,2010-2011,ANA
2,5v5,82,3845.2667,3022.00,3793.00,44.34,2356.00,2870.00,45.08,1701.00,...,86.00,87.00,49.71,17.59,84.52,7.82,92.31,1.001,2010-2011,ANA
3,5v5 Score & Venue Adjusted,82,3845.2667,3000.72,3748.88,44.46,2340.52,2845.97,45.13,1685.89,...,86.04,86.26,49.94,17.82,84.50,7.86,92.26,1.001,2010-2011,ANA
4,5v4 Power Play,82,429.4000,730.00,95.00,88.48,585.00,80.00,87.97,427.00,...,39.00,3.00,92.86,25.83,82.35,13.82,90.91,1.047,2010-2011,ANA


In [92]:
import pandas as pd
import glob
import os
# Adjust the pattern if your files are in a folder:
# example: 'data/nst_team_season_*.csv'
csv_files = glob.glob("nst_team_season_*.csv")

print("Found CSV files:", csv_files)

dfs = []

for file in csv_files:
    print(f"Loading {file}...")
    df = pd.read_csv(file)
    dfs.append(df)

# Combine all seasons into one DataFrame
nst_all = pd.concat(dfs, ignore_index=True)

print("Combined shape:", nst_all.shape)


Found CSV files: ['nst_team_season_2015_2016.csv', 'nst_team_season_2014_2015.csv', 'nst_team_season_2024_2025.csv', 'nst_team_season_2023_2024.csv', 'nst_team_season_2017_2018.csv', 'nst_team_season_2012_2013.csv', 'nst_team_season_2022_2023.csv', 'nst_team_season_2013_2014.csv', 'nst_team_season_2018_2019.csv', 'nst_team_season_2019_2020.csv', 'nst_team_season_2016_2017.csv', 'nst_team_season_2020_2021.csv', 'nst_team_season_2021_2022.csv', 'nst_team_season_2011_2012.csv', 'nst_team_season_2010_2011.csv']
Loading nst_team_season_2015_2016.csv...
Loading nst_team_season_2014_2015.csv...
Loading nst_team_season_2024_2025.csv...
Loading nst_team_season_2023_2024.csv...
Loading nst_team_season_2017_2018.csv...
Loading nst_team_season_2012_2013.csv...
Loading nst_team_season_2022_2023.csv...
Loading nst_team_season_2013_2014.csv...
Loading nst_team_season_2018_2019.csv...
Loading nst_team_season_2019_2020.csv...
Loading nst_team_season_2016_2017.csv...
Loading nst_team_season_2020_2021.cs

In [94]:
# Convert Season string ("2010-2011") → start_year for sorting
nst_all["Season_Start"] = nst_all["Season"].str.split("-").str[0].astype(int)

# Sort by season then team
nst_all = nst_all.sort_values(["Season_Start", "Team"]).reset_index(drop=True)

# Drop accidental duplicates
nst_all = nst_all.drop_duplicates()

# Drop helper column if you want
nst_all = nst_all.drop(columns=["Season_Start"])

nst_all.head()


,Situation,GP,TOI,CF,CA,CFPct,FF,FA,FFPct,SF,...,HDGF,HDGA,HDGFPct,HDSHPct,HDSVPct,SHPct,SVPct,PDO,Season,Team
0,All Situations,82,4992.7167,4093.00,5042.00,44.81,3219.00,3794.00,45.90,2334.00,...,143.00,131.00,52.19,20.52,82.56,10.07,91.20,1.013,2010-2011,ANA
1,Even Strength,82,4039.4167,3221.00,4010.00,44.54,2509.00,3038.00,45.23,1812.00,...,95.00,93.00,50.53,18.45,84.32,8.72,91.99,1.007,2010-2011,ANA
2,5v5,82,3845.2667,3022.00,3793.00,44.34,2356.00,2870.00,45.08,1701.00,...,86.00,87.00,49.71,17.59,84.52,7.82,92.31,1.001,2010-2011,ANA
3,5v5 Score & Venue Adjusted,82,3845.2667,3000.72,3748.88,44.46,2340.52,2845.97,45.13,1685.89,...,86.04,86.26,49.94,17.82,84.50,7.86,92.26,1.001,2010-2011,ANA
4,5v4 Power Play,82,429.4000,730.00,95.00,88.48,585.00,80.00,87.97,427.00,...,39.00,3.00,92.86,25.83,82.35,13.82,90.91,1.047,2010-2011,ANA


In [96]:
nst_all.to_csv("nst_team_season_all_years.csv", index=False)
print("Saved combined file: nst_team_season_all_years.csv")


Saved combined file: nst_team_season_all_years.csv


In [166]:
import pandas as pd

# Game logs (Hockey-Reference output)
games_df = pd.read_csv("nhl_games_2010_2024.csv", parse_dates=["game_date"])

# NST team-season stats (the big combined file you just built)
nst_all = pd.read_csv("nst_team_season_all_years.csv")


In [168]:
import pandas as pd

# If you loaded from CSV, reload here (or skip if already in memory)
# games_df = pd.read_csv("nhl_games_2013_2024.csv", parse_dates=["game_date"])
# nst_all = pd.read_csv("nst_team_season_all_years.csv")

# Ensure season is integer (NHL season end year, e.g., 2019 for 2018-19)
print("games_df columns:", games_df.columns)

if "season" not in games_df.columns:
    # fallback: derive from game_date.year (adjust if needed)
    games_df["game_date"] = pd.to_datetime(games_df["game_date"])
    games_df["season"] = games_df["game_date"].dt.year

# Map team names -> codes
TEAM_NAME_TO_CODE = {
    "Anaheim Ducks": "ANA",
    "Arizona Coyotes": "ARI",
    "Boston Bruins": "BOS",
    "Buffalo Sabres": "BUF",
    "Carolina Hurricanes": "CAR",
    "Columbus Blue Jackets": "CBJ",
    "Calgary Flames": "CGY",
    "Chicago Blackhawks": "CHI",
    "Colorado Avalanche": "COL",
    "Dallas Stars": "DAL",
    "Detroit Red Wings": "DET",
    "Edmonton Oilers": "EDM",
    "Florida Panthers": "FLA",
    "Los Angeles Kings": "LAK",
    "Minnesota Wild": "MIN",
    "Montreal Canadiens": "MTL",
    "Montréal Canadiens": "MTL",
    "New Jersey Devils": "NJD",
    "Nashville Predators": "NSH",
    "New York Islanders": "NYI",
    "New York Rangers": "NYR",
    "Ottawa Senators": "OTT",
    "Philadelphia Flyers": "PHI",
    "Pittsburgh Penguins": "PIT",
    "San Jose Sharks": "SJS",
    "St. Louis Blues": "STL",
    "Tampa Bay Lightning": "TBL",
    "Toronto Maple Leafs": "TOR",
    "Vancouver Canucks": "VAN",
    "Vegas Golden Knights": "VGK",
    "Washington Capitals": "WSH",
    "Winnipeg Jets": "WPG",
    "Phoenix Coyotes": "ARI",  # older name
    "Atlanta Thrashers": "ATL",  # if they appear in your range
    "Seattle Kraken": "SEA",
}

games_df["home_team_code"] = games_df["home_team"].map(TEAM_NAME_TO_CODE)
games_df["away_team_code"] = games_df["away_team"].map(TEAM_NAME_TO_CODE)

print("Missing home_team_code:", games_df["home_team_code"].isna().sum())
print("Missing away_team_code:", games_df["away_team_code"].isna().sum())


games_df columns: Index(['game_id', 'season', 'game_date', 'away_team', 'away_goals',
       'home_team', 'home_goals', 'home_win', 'home_win_margin'],
      dtype='object')
Missing home_team_code: 0
Missing away_team_code: 0


In [170]:
# games_df["home_team_code"] and ["away_team_code"] already created with TEAM_NAME_TO_CODE
print("Missing home_team_code:", games_df["home_team_code"].isna().sum())
print("Missing away_team_code:", games_df["away_team_code"].isna().sum())


Missing home_team_code: 0
Missing away_team_code: 0


In [172]:
# 1) Create NST 'season_end' and 'season_for_games' (shifted)
nst_all["season_end"] = nst_all["Season"].str.split("-").str[1].astype(int)
nst_all["season_for_games"] = nst_all["season_end"] - 1

print("NST season_end:", sorted(nst_all["season_end"].unique())[:5], "...",
      sorted(nst_all["season_end"].unique())[-5:])
print("NST season_for_games:", sorted(nst_all["season_for_games"].unique())[:5], "...",
      sorted(nst_all["season_for_games"].unique())[-5:])

# 2) Find situation column and choose one row per team-season
situation_candidates = [
    c for c in nst_all.columns
    if "sit" in c.lower() or "situation" in c.lower()
]
print("Situation candidates:", situation_candidates)

if situation_candidates:
    situation_col = situation_candidates[0]
    # Prefer 'All' situations if available
    mask_all = nst_all[situation_col].astype(str).str.contains("All", case=False)
    nst_filtered = nst_all[mask_all].copy()
    if nst_filtered.empty:
        # Fallback: first row per season_for_games + Team
        nst_filtered = (
            nst_all.sort_values(["season_for_games", "Team"])
            .drop_duplicates(subset=["season_for_games", "Team"], keep="first")
            .copy()
        )
else:
    # No situation column; just one row per season_for_games + Team
    nst_filtered = (
        nst_all.sort_values(["season_for_games", "Team"])
        .drop_duplicates(subset=["season_for_games", "Team"], keep="first")
        .copy()
    )

# 3) Build feature set
feature_cols = [
    c for c in nst_filtered.columns
    if c not in ["Season", "Team", "season_end", "season_for_games"] + situation_candidates
]

nst_features = nst_filtered[["season_for_games", "Team"] + feature_cols].copy()
nst_features = nst_features.rename(columns={"season_for_games": "season"})

print("nst_features sample:")
print(nst_features.head())
print("NST seasons for merge:", sorted(nst_features["season"].unique()))


NST season_end: [2011, 2012, 2013, 2014, 2015] ... [2021, 2022, 2023, 2024, 2025]
NST season_for_games: [2010, 2011, 2012, 2013, 2014] ... [2020, 2021, 2022, 2023, 2024]
Situation candidates: ['Situation']
nst_features sample:
    season Team  GP        TOI      CF      CA  CFPct      FF      FA  FFPct  \
0     2010  ANA  82  4992.7167  4093.0  5042.0  44.81  3219.0  3794.0  45.90   
6     2010  ARI  82  4995.4833  4658.0  4827.0  49.11  3518.0  3771.0  48.26   
12    2010  BOS  82  4980.0667  4923.0  4791.0  50.68  3644.0  3619.0  50.17   
18    2010  BUF  82  4999.0833  4689.0  4690.0  49.99  3567.0  3522.0  50.32   
24    2010  CAR  82  4996.7000  4935.0  5083.0  49.26  3562.0  3880.0  47.86   

    ...   HDSA  HDSFPct   HDGF   HDGA  HDGFPct  HDSHPct  HDSVPct  SHPct  \
0   ...  751.0    48.14  143.0  131.0    52.19    20.52    82.56  10.07   
6   ...  722.0    46.76  122.0  124.0    49.59    19.24    82.83   9.09   
12  ...  682.0    49.26  134.0  108.0    55.37    20.24    84.16   

In [174]:
# --- Home NST features ---
home_nst = nst_features.rename(
    columns={
        "Team": "home_team_code",
        **{c: f"home_{c}" for c in feature_cols}
    }
)

# --- Away NST features ---
away_nst = nst_features.rename(
    columns={
        "Team": "away_team_code",
        **{c: f"away_{c}" for c in feature_cols}
    }
)

print("home_nst columns:", home_nst.columns[:10])
print("away_nst columns:", away_nst.columns[:10])

# --- Merge onto games_df ---
games_with_home = games_df.merge(
    home_nst,
    on=["season", "home_team_code"],
    how="left",
)

games_with_nst = games_with_home.merge(
    away_nst,
    on=["season", "away_team_code"],
    how="left",
)

print("Merged shape:", games_with_nst.shape)
games_with_nst.head()


home_nst columns: Index(['season', 'home_team_code', 'home_GP', 'home_TOI', 'home_CF', 'home_CA',
       'home_CFPct', 'home_FF', 'home_FA', 'home_FFPct'],
      dtype='object')
away_nst columns: Index(['season', 'away_team_code', 'away_GP', 'away_TOI', 'away_CF', 'away_CA',
       'away_CFPct', 'away_FF', 'away_FA', 'away_FFPct'],
      dtype='object')
Merged shape: (19118, 95)


,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin,home_team_code,...,away_HDSA,away_HDSFPct,away_HDGF,away_HDGA,away_HDGFPct,away_HDSHPct,away_HDSVPct,away_SHPct,away_SVPct,away_PDO
0,20091001_WashingtonCapitals_@_BostonBruins,2010,2009-10-01,Washington Capitals,4,Boston Bruins,1,0,-3,BOS,...,619.0,52.64,112.0,109.0,50.68,16.28,82.39,8.53,91.96,1.005
1,20091001_VancouverCanucks_@_CalgaryFlames,2010,2009-10-01,Vancouver Canucks,3,Calgary Flames,5,1,2,CGY,...,734.0,49.52,138.0,101.0,57.74,19.17,86.24,9.83,92.71,1.025
2,20091001_SanJoseSharks_@_ColoradoAvalanche,2010,2009-10-01,San Jose Sharks,2,Colorado Avalanche,5,1,3,COL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20091001_MontrealCanadiens_@_TorontoMapleLeafs,2010,2009-10-01,Montreal Canadiens,4,Toronto Maple Leafs,3,0,-1,TOR,...,730.0,47.60,107.0,124.0,46.32,16.14,83.01,8.19,91.88,1.001
4,20091002_PhiladelphiaFlyers_@_CarolinaHurricanes,2010,2009-10-02,Philadelphia Flyers,2,Carolina Hurricanes,0,0,-2,CAR,...,670.0,54.14,154.0,108.0,58.78,19.47,83.88,9.83,91.26,1.011


In [176]:
home_nst_cols = [c for c in games_with_nst.columns if c.startswith("home_")]
print("Example home NST columns:", home_nst_cols[:10])

if home_nst_cols:
    example_col = home_nst_cols[0]
    coverage = games_with_nst[example_col].notna().mean()
    print(f"Fraction of games with NST data for {example_col}: {coverage:.3f}")


Example home NST columns: ['home_team', 'home_goals', 'home_win', 'home_win_margin', 'home_team_code', 'home_GP', 'home_TOI', 'home_CF', 'home_CA', 'home_CFPct']
Fraction of games with NST data for home_team: 1.000


In [178]:
games_with_nst.to_csv("nhl_games_with_nst_season_features.csv", index=False)


Scrape NST game by game

In [198]:
df = pd.read_csv('all_teams.csv')
df.head()

,team,season,name,gameId,playerTeam,opposingTeam,home_or_away,gameDate,position,situation,...,unblockedShotAttemptsAgainst,scoreAdjustedUnblockedShotAttemptsAgainst,dZoneGiveawaysAgainst,xGoalsFromxReboundsOfShotsAgainst,xGoalsFromActualReboundsOfShotsAgainst,reboundxGoalsAgainst,totalShotCreditAgainst,scoreAdjustedTotalShotCreditAgainst,scoreFlurryAdjustedTotalShotCreditAgainst,playoffGame
0,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,other,...,1.0,1.000,0.0,0.017,0.000,0.000,0.037,0.037,0.037,0
1,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,all,...,31.0,30.369,5.0,0.396,0.168,0.168,2.917,2.833,2.714,0
2,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,5on5,...,20.0,19.369,3.0,0.237,0.168,0.168,1.862,1.777,1.665,0
3,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,4on5,...,9.0,9.000,1.0,0.124,0.000,0.000,0.795,0.795,0.789,0
4,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,5on4,...,1.0,1.000,1.0,0.019,0.000,0.000,0.224,0.224,0.224,0


In [202]:
df.columns.values

array(['team', 'season', 'name', 'gameId', 'playerTeam', 'opposingTeam',
       'home_or_away', 'gameDate', 'position', 'situation',
       'xGoalsPercentage', 'corsiPercentage', 'fenwickPercentage',
       'iceTime', 'xOnGoalFor', 'xGoalsFor', 'xReboundsFor', 'xFreezeFor',
       'xPlayStoppedFor', 'xPlayContinuedInZoneFor',
       'xPlayContinuedOutsideZoneFor', 'flurryAdjustedxGoalsFor',
       'scoreVenueAdjustedxGoalsFor', 'flurryScoreVenueAdjustedxGoalsFor',
       'shotsOnGoalFor', 'missedShotsFor', 'blockedShotAttemptsFor',
       'shotAttemptsFor', 'goalsFor', 'reboundsFor', 'reboundGoalsFor',
       'freezeFor', 'playStoppedFor', 'playContinuedInZoneFor',
       'playContinuedOutsideZoneFor', 'savedShotsOnGoalFor',
       'savedUnblockedShotAttemptsFor', 'penaltiesFor',
       'penalityMinutesFor', 'faceOffsWonFor', 'hitsFor', 'takeawaysFor',
       'giveawaysFor', 'lowDangerShotsFor', 'mediumDangerShotsFor',
       'highDangerShotsFor', 'lowDangerxGoalsFor',
       'mediumDa

Get historical NHL odds from https://the-odds-api.com/

In [204]:
import requests
import time
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

API_KEY = "aba4527f4425936acadb110a37a9918c"  

BASE_URL = "https://api.the-odds-api.com/v4"

def fetch_odds(
    sport_key: str,
    regions: str = "us",
    markets: str = "h2h",
    odds_format: str = "american",
    date: str | None = None,
    bookmaker: str | None = None,
):
    """
    Fetch odds from The Odds API.
    sport_key example for NHL: 'icehockey_nhl'
    regions: 'us', 'eu', 'uk', etc.
    markets: 'h2h' (moneyline), 'spreads', 'totals'
    date: for historical odds, you’ll use their 'date' or 'from' param per docs.
    """
    params = {
        "apiKey": API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
    }
    if bookmaker:
        params["bookmakers"] = bookmaker
    if date:
        params["date"] = date  # or the correct param name per their docs

    url = f"{BASE_URL}/sports/{sport_key}/odds"
    logging.info("Requesting: %s", url)
    resp = requests.get(url, params=params, timeout=20)

    if resp.status_code != 200:
        logging.error("Error %s: %s", resp.status_code, resp.text[:200])
        return None, resp

    data = resp.json()
    return data, resp



In [206]:
from datetime import datetime, timedelta

def daterange(start_date: datetime, end_date: datetime):
    cur = start_date
    while cur <= end_date:
        yield cur
        cur += timedelta(days=1)

def fetch_nhl_odds_for_range(
    start_date: str,
    end_date: str,
    sport_key: str = "icehockey_nhl",
    region: str = "us",
    bookmaker: str | None = None,
    sleep_sec: float = 1.0,
):
    """
    Fetch NHL odds for each day in [start_date, end_date].
    start_date/end_date: 'YYYY-MM-DD'
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    all_rows = []

    for d in daterange(start, end):
        date_str = d.strftime("%Y-%m-%d")
        logging.info("Fetching odds for date %s", date_str)

        data, resp = fetch_odds(
            sport_key=sport_key,
            regions=region,
            markets="h2h",
            odds_format="american",
            date=date_str,
            bookmaker=bookmaker,
        )

        # If the request failed, skip
        if data is None:
            continue

        # Check remaining credits from headers if provided
        remain = resp.headers.get("x-requests-remaining") or resp.headers.get("x-requests-remaining-month")
        logging.info("Remaining request credits (if provided): %s", remain)

        # Flatten each game into rows
        for game in data:
            game_id = game.get("id")
            commence_time = game.get("commence_time")  # ISO timestamp
            home_team = game.get("home_team")
            away_team = None
            if game.get("away_team"):
                away_team = game["away_team"]
            else:
                # some APIs use team list instead
                teams = game.get("teams", [])
                if len(teams) == 2:
                    # assume [away, home] or similar – check docs
                    away_team = teams[0]

            for book in game.get("bookmakers", []):
                bk_key = book.get("key")
                for market in book.get("markets", []):
                    if market.get("key") != "h2h":
                        continue
                    for outcome in market.get("outcomes", []):
                        row = {
                            "api_game_id": game_id,
                            "commence_time": commence_time,
                            "date": date_str,
                            "bookmaker": bk_key,
                            "market": market.get("key"),
                            "team": outcome.get("name"),
                            "price": outcome.get("price"),  # American odds
                        }
                        all_rows.append(row)

        time.sleep(sleep_sec)

    if not all_rows:
        return pd.DataFrame()

    return pd.DataFrame(all_rows)


In [208]:
# Gets 2023-2024 odds from pinnacle 274 credits

nhl_odds_2023_24 = fetch_nhl_odds_for_range(
    start_date="2023-10-01",
    end_date="2024-06-30",
    bookmaker="pinnacle",  # or None for all
    sleep_sec=1.0
)

nhl_odds_2023_24.to_csv("nhl_odds_2023_24_pinnacle.csv", index=False)


2025-11-15 15:02:19,490 | INFO | Fetching odds for date 2023-10-01
2025-11-15 15:02:19,492 | INFO | Requesting: https://api.the-odds-api.com/v4/sports/icehockey_nhl/odds
2025-11-15 15:02:19,865 | INFO | Remaining request credits (if provided): 499
2025-11-15 15:02:20,868 | INFO | Fetching odds for date 2023-10-02
2025-11-15 15:02:20,869 | INFO | Requesting: https://api.the-odds-api.com/v4/sports/icehockey_nhl/odds
2025-11-15 15:02:21,416 | INFO | Remaining request credits (if provided): 498
2025-11-15 15:02:22,420 | INFO | Fetching odds for date 2023-10-03
2025-11-15 15:02:22,421 | INFO | Requesting: https://api.the-odds-api.com/v4/sports/icehockey_nhl/odds
2025-11-15 15:02:22,761 | INFO | Remaining request credits (if provided): 497
2025-11-15 15:02:23,768 | INFO | Fetching odds for date 2023-10-04
2025-11-15 15:02:23,770 | INFO | Requesting: https://api.the-odds-api.com/v4/sports/icehockey_nhl/odds
2025-11-15 15:02:24,116 | INFO | Remaining request credits (if provided): 496
2025-11-

In [210]:
nhl_odds_2023_24.head()

,api_game_id,commence_time,date,bookmaker,market,team,price
0,4a2b9236eb60f41001d7fed056bb6904,2025-11-15T22:10:00Z,2023-10-01,pinnacle,h2h,Florida Panthers,154
1,4a2b9236eb60f41001d7fed056bb6904,2025-11-15T22:10:00Z,2023-10-01,pinnacle,h2h,Tampa Bay Lightning,-182
2,f1719616041b422672fcbe179f2aa2f7,2025-11-15T23:05:00Z,2023-10-01,pinnacle,h2h,Anaheim Ducks,103
3,f1719616041b422672fcbe179f2aa2f7,2025-11-15T23:05:00Z,2023-10-01,pinnacle,h2h,Minnesota Wild,-113
4,b287e6951f76dbeec045d3274c43ea9b,2025-11-16T00:00:00Z,2023-10-01,pinnacle,h2h,Boston Bruins,136


In [216]:
import pandas as pd

games = pd.read_csv("nhl_games_2010_2024.csv")
games["game_date"] = pd.to_datetime(games["game_date"])

cutoff = pd.Timestamp("2020-06-06")

unique_dates = (
    games.loc[games["game_date"] >= cutoff, "game_date"]
    .dt.normalize()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

len(unique_dates), unique_dates[:5]



(903,
 [Timestamp('2020-08-01 00:00:00'),
  Timestamp('2020-08-02 00:00:00'),
  Timestamp('2020-08-03 00:00:00'),
  Timestamp('2020-08-04 00:00:00'),
  Timestamp('2020-08-05 00:00:00')])

In [218]:
import requests
import time
import logging
from datetime import datetime
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

API_KEY = "aba4527f4425936acadb110a37a9918c" 
SPORT_KEY = "icehockey_nhl"
BASE_HIST_URL = f"https://api.the-odds-api.com/v4/historical/sports/{SPORT_KEY}/odds"

BOOKMAKERS_US = ["fanduel", "draftkings"]
BOOKMAKERS_EU = ["pinnacle"]

def fetch_historical_snapshot(
    snapshot_iso: str,
    region: str,
    bookmakers: list[str],
    market: str = "h2h",
):
    """
    Fetch one historical odds snapshot for NHL for a given timestamp & region.
    snapshot_iso: ISO8601 string, e.g. '2023-10-10T23:59:59Z'
    region: 'us' or 'eu'
    bookmakers: list of bookmaker keys in that region
    """
    params = {
        "apiKey": API_KEY,
        "regions": region,
        "markets": market,
        "oddsFormat": "american",
        "date": snapshot_iso,
        "bookmakers": ",".join(bookmakers),
    }

    logging.info("GET %s region=%s date=%s", BASE_HIST_URL, region, snapshot_iso)
    resp = requests.get(BASE_HIST_URL, params=params, timeout=30)
    if resp.status_code != 200:
        logging.error("Status %s: %s", resp.status_code, resp.text[:200])
        return None

    data = resp.json()

    # Historical wrapper includes snapshot metadata
    snapshot_ts = None
    events = data
    if isinstance(data, dict) and "timestamp" in data and "data" in data:
        snapshot_ts = data.get("timestamp")
        events = data.get("data", [])

    rows = []

    for event in events:
        event_id = event.get("id")
        commence_time = event.get("commence_time")
        home_team = event.get("home_team")
        away_team = event.get("away_team")

        for bk in event.get("bookmakers", []):
            bk_key = bk.get("key")
            for market_obj in bk.get("markets", []):
                if market_obj.get("key") != market:
                    continue
                for outcome in market_obj.get("outcomes", []):
                    rows.append(
                        {
                            "snapshot_ts": snapshot_ts,
                            "snapshot_request_ts": snapshot_iso,
                            "region": region,
                            "sport_key": SPORT_KEY,
                            "event_id": event_id,
                            "commence_time": commence_time,
                            "home_team": home_team,
                            "away_team": away_team,
                            "bookmaker": bk_key,
                            "market": market,
                            "outcome_team": outcome.get("name"),
                            "price": outcome.get("price"),
                        }
                    )

    return rows


In [220]:
import requests
import time
import logging
from datetime import datetime, timedelta
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

API_KEY = "fd89ad7935c3f3851f9827d2bdaef90f"  
SPORT_KEY = "icehockey_nhl"
BASE_HIST_URL = f"https://api.the-odds-api.com/v4/historical/sports/{SPORT_KEY}/odds"

# Sharp books (EU region)
BOOKMAKERS_EU = ["pinnacle", "betfair_ex_eu"]

# Soft books (US region)
BOOKMAKERS_US = ["draftkings", "fanduel"]

# Markets: moneyline (h2h), spreads, totals
MARKETS = "h2h,spreads,totals"


In [222]:
games = pd.read_csv("nhl_games_2010_2024.csv")
games["game_date"] = pd.to_datetime(games["game_date"])

# The Odds API history starts 2020-06-06
cutoff = pd.Timestamp("2020-06-06")
unique_dates = (
    games.loc[games["game_date"] >= cutoff, "game_date"]
         .dt.normalize()
         .drop_duplicates()
         .sort_values()
         .tolist()
)

len(unique_dates), unique_dates[:5]


(903,
 [Timestamp('2020-08-01 00:00:00'),
  Timestamp('2020-08-02 00:00:00'),
  Timestamp('2020-08-03 00:00:00'),
  Timestamp('2020-08-04 00:00:00'),
  Timestamp('2020-08-05 00:00:00')])

In [224]:
def fetch_historical_snapshot(
    snapshot_iso: str,
    region: str,
    bookmakers: list[str],
    markets: str = MARKETS,
    odds_format: str = "american",
):
    """
    Fetch one historical odds snapshot for NHL for a given timestamp & region.

    snapshot_iso: ISO8601 string, e.g. '2023-10-10T23:59:59Z'
    region: 'us' or 'eu'
    bookmakers: list of bookmaker keys in that region
    markets: 'h2h,spreads,totals'
    """
    params = {
        "apiKey": API_KEY,
        "regions": region,
        "markets": markets,
        "oddsFormat": odds_format,
        "date": snapshot_iso,
        "bookmakers": ",".join(bookmakers),
    }

    logging.info("GET %s region=%s date=%s", BASE_HIST_URL, region, snapshot_iso)
    resp = requests.get(BASE_HIST_URL, params=params, timeout=30)

    if resp.status_code != 200:
        logging.error("Status %s: %s", resp.status_code, resp.text[:200])
        return []

    data = resp.json()

    # Some tiers wrap in {"timestamp": ..., "data": [...]}
    snapshot_ts = None
    events = data
    if isinstance(data, dict) and "timestamp" in data and "data" in data:
        snapshot_ts = data.get("timestamp")
        events = data.get("data", [])

    rows = []

    for event in events:
        event_id = event.get("id")
        commence_time = event.get("commence_time")
        home_team = event.get("home_team")
        away_team = event.get("away_team")

        for bk in event.get("bookmakers", []):
            bk_key = bk.get("key")
            for mkt in bk.get("markets", []):
                market_key = mkt.get("key")  # 'h2h', 'spreads', 'totals', etc.
                if market_key not in ["h2h", "spreads", "totals"]:
                    continue

                for outcome in mkt.get("outcomes", []):
                    # For h2h: outcome name = team
                    # For spreads/totals: outcome has 'point' field too.
                    row = {
                        "snapshot_ts": snapshot_ts,
                        "snapshot_request_ts": snapshot_iso,
                        "region": region,
                        "sport_key": SPORT_KEY,
                        "event_id": event_id,
                        "commence_time": commence_time,
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmaker": bk_key,
                        "market": market_key,           # h2h / spreads / totals
                        "outcome_name": outcome.get("name"),
                        "price": outcome.get("price"),  # American odds
                        "point": outcome.get("point"),  # spread or total line (None for h2h)
                    }
                    rows.append(row)

    return rows


In [226]:
all_rows = []

for d in unique_dates:
    iso_ts = d.strftime("%Y-%m-%dT23:59:59Z")
    logging.info("=== Date %s ===", iso_ts)

    # US region: DraftKings + FanDuel
    us_rows = fetch_historical_snapshot(
        snapshot_iso=iso_ts,
        region="us",
        bookmakers=BOOKMAKERS_US,
    )
    all_rows.extend(us_rows)
    time.sleep(0.3)  # be polite

    # EU region: Pinnacle + Betfair Exchange EU
    eu_rows = fetch_historical_snapshot(
        snapshot_iso=iso_ts,
        region="eu",
        bookmakers=BOOKMAKERS_EU,
    )
    all_rows.extend(eu_rows)
    time.sleep(0.3)


2025-11-21 11:16:33,597 | INFO | === Date 2020-08-01T23:59:59Z ===
2025-11-21 11:16:33,598 | INFO | GET https://api.the-odds-api.com/v4/historical/sports/icehockey_nhl/odds region=us date=2020-08-01T23:59:59Z
2025-11-21 11:16:34,738 | INFO | GET https://api.the-odds-api.com/v4/historical/sports/icehockey_nhl/odds region=eu date=2020-08-01T23:59:59Z
2025-11-21 11:16:35,556 | INFO | === Date 2020-08-02T23:59:59Z ===
2025-11-21 11:16:35,559 | INFO | GET https://api.the-odds-api.com/v4/historical/sports/icehockey_nhl/odds region=us date=2020-08-02T23:59:59Z
2025-11-21 11:16:36,671 | INFO | GET https://api.the-odds-api.com/v4/historical/sports/icehockey_nhl/odds region=eu date=2020-08-02T23:59:59Z
2025-11-21 11:16:37,798 | INFO | === Date 2020-08-03T23:59:59Z ===
2025-11-21 11:16:37,800 | INFO | GET https://api.the-odds-api.com/v4/historical/sports/icehockey_nhl/odds region=us date=2020-08-03T23:59:59Z
2025-11-21 11:16:38,870 | INFO | GET https://api.the-odds-api.com/v4/historical/sports/ic

KeyboardInterrupt: 

In [229]:
hist_odds_df = pd.DataFrame(all_rows)
print(hist_odds_df.head())
print(hist_odds_df["market"].value_counts())
print(hist_odds_df["bookmaker"].value_counts())

hist_odds_df.to_csv(
    "nhl_historical_odds_2020plus_pinnacle_betfair_dk_fd_ml_spreads_totals.csv",
    index=False
)
print("Saved NHL historical odds CSV.")


            snapshot_ts   snapshot_request_ts region      sport_key  \
0  2020-08-01T23:55:00Z  2020-08-01T23:59:59Z     us  icehockey_nhl   
1  2020-08-01T23:55:00Z  2020-08-01T23:59:59Z     us  icehockey_nhl   
2  2020-08-01T23:55:00Z  2020-08-01T23:59:59Z     us  icehockey_nhl   
3  2020-08-01T23:55:00Z  2020-08-01T23:59:59Z     us  icehockey_nhl   
4  2020-08-01T23:55:00Z  2020-08-01T23:59:59Z     us  icehockey_nhl   

                           event_id         commence_time  \
0  b2020f52aad491b409755933742b3603  2020-08-02T02:30:00Z   
1  b2020f52aad491b409755933742b3603  2020-08-02T02:30:00Z   
2  730b76ebabbb9841b784370e52c3506b  2020-08-02T18:00:00Z   
3  730b76ebabbb9841b784370e52c3506b  2020-08-02T18:00:00Z   
4  d70b6475db17c5bd9ca91bce63651878  2020-08-03T00:00:00Z   

             home_team              away_team   bookmaker market  \
0       Calgary Flames          Winnipeg Jets  draftkings    h2h   
1       Calgary Flames          Winnipeg Jets  draftkings    h2h   
2 

In [233]:
hist_odds_df.sample(50)

,snapshot_ts,snapshot_request_ts,region,sport_key,event_id,commence_time,home_team,away_team,bookmaker,market,outcome_name,price,point
28104,2021-10-15T23:55:00Z,2021-10-15T23:59:59Z,us,icehockey_nhl,574e3110bc3b104cc11edb19aa9ff4d2,2021-10-15T23:00:00Z,Philadelphia Flyers,Vancouver Canucks,draftkings,h2h,Philadelphia Flyers,-385,NaN
20174,2021-04-22T23:55:00Z,2021-04-22T23:59:59Z,us,icehockey_nhl,3c9b113fb3afbf042f2ad4022a1bf835,2021-04-22T23:00:00Z,New York Islanders,Washington Capitals,fanduel,h2h,New York Islanders,-145,NaN
14281,2021-03-25T23:55:00Z,2021-03-25T23:59:59Z,eu,icehockey_nhl,3dd6f022a44135d78d5ff0928790a8a0,2021-03-25T23:00:00Z,Washington Capitals,New Jersey Devils,pinnacle,spreads,Washington Capitals,156,-1.5
14619,2021-03-27T23:55:00Z,2021-03-27T23:59:59Z,us,icehockey_nhl,e8b83bbe822288cc7e29f35bba09d353,2021-03-28T02:10:00Z,Calgary Flames,Winnipeg Jets,draftkings,totals,Under,-123,6.0
27856,2021-10-13T23:55:00Z,2021-10-13T23:59:59Z,eu,icehockey_nhl,d8fbfa25f3fd3c82ebb4e8bfa13c0b51,2021-10-14T02:00:00Z,Colorado Avalanche,Chicago Blackhawks,pinnacle,spreads,Chicago Blackhawks,-147,1.5
26091,2021-05-22T23:55:00Z,2021-05-22T23:59:59Z,eu,icehockey_nhl,047c6f70104d34fb0b06537d0e417bf2,2021-05-22T23:00:00Z,Toronto Maple Leafs,Montréal Canadiens,pinnacle,spreads,Toronto Maple Leafs,162,-1.5
38199,2021-12-08T23:55:00Z,2021-12-08T23:59:59Z,us,icehockey_nhl,353a541eb7b44d4548213ddd6a55fa5d,2021-12-09T00:00:00Z,New York Rangers,Colorado Avalanche,fanduel,spreads,New York Rangers,-182,1.5
12225,2021-03-15T23:55:00Z,2021-03-15T23:59:59Z,us,icehockey_nhl,ced956583744ae37c3d7a3d7ab7600ac,2021-03-16T23:00:00Z,New Jersey Devils,Buffalo Sabres,draftkings,h2h,New Jersey Devils,-141,NaN
3941,2021-01-28T23:55:00Z,2021-01-28T23:59:59Z,eu,icehockey_nhl,bc9369f536e18f6c5c26bedebf43bcee,2021-01-29T00:00:00Z,Buffalo Sabres,New York Rangers,pinnacle,totals,Under,-108,6.0
6605,2021-02-14T23:55:00Z,2021-02-14T23:59:59Z,us,icehockey_nhl,45af479bde9eb6108af3dbafbd2b6fe8,2021-02-16T00:00:00Z,Tampa Bay Lightning,Florida Panthers,draftkings,spreads,Tampa Bay Lightning,155,-1.5


In [247]:
import requests
import pandas as pd
from datetime import datetime
import time
import os

API_KEY = "fd89ad7935c3f3851f9827d2bdaef90f"   # <-- put your key here

SPORT_KEY = "icehockey_nhl"
BASE_URL = f"https://api.the-odds-api.com/v4/historical/sports/{SPORT_KEY}/odds"

MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "american"

# US books
BOOKS_US = ["draftkings", "fanduel"]
# EU books
BOOKS_EU = ["pinnacle", "betfair_ex_eu"]

# ---------------------------------------
# 1) Load advanced stats to get true game dates
# ---------------------------------------
games = pd.read_csv("games_with_full_roll", low_memory=False)
games["game_date"] = pd.to_datetime(games["game_date"]).dt.date

# The advanced stats DF spans 2010–2024.
# Your previous odds file ends on 2022-02-15, so we continue from there.
dates_needed = sorted({
    d for d in games["game_date"]
    if d > datetime(2022, 2, 15).date()
})

print("Total game dates to fetch:", len(dates_needed))

# ---------------------------------------
# 2) Resume-safe output file
# ---------------------------------------
out_path = "odds_2022_02_16_to_present_four_books.csv"

if os.path.exists(out_path):
    existing = pd.read_csv(out_path, low_memory=False)
    existing_dates = set(pd.to_datetime(existing["game_date"]).dt.date)
    dates_needed = [d for d in dates_needed if d not in existing_dates]
    print("Resuming. Remaining dates:", len(dates_needed))
else:
    existing_dates = set()

# ---------------------------------------
# 3) Helper scraping function
# ---------------------------------------
def scrape_date_region(date_obj, region, books):
    snapshot_iso = f"{date_obj.isoformat()}T23:59:59Z"

    params = {
        "apiKey": API_KEY,
        "regions": region,
        "markets": MARKETS,
        "oddsFormat": ODDS_FORMAT,
        "date": snapshot_iso,
        "bookmakers": ",".join(books),
    }

    print(f"[{date_obj}] region={region} books={','.join(books)}")

    try:
        resp = requests.get(BASE_URL, params=params, timeout=30)

        if resp.status_code != 200:
            print("  ERROR:", resp.status_code, resp.text[:200])
            return []

        data = resp.json()

        # API format wrapper compatibility
        if isinstance(data, dict) and "data" in data:
            events = data["data"]
            snapshot_ts = data.get("timestamp")
        else:
            events = data
            snapshot_ts = None

        rows = []

        for event in events:
            event_id = event.get("id")
            commence = event.get("commence_time")
            home = event.get("home_team")
            away = event.get("away_team")

            for bk in event.get("bookmakers", []):
                bk_key = bk.get("key")
                if bk_key not in books:
                    continue

                last_update = bk.get("last_update", snapshot_ts)

                for mkt in bk.get("markets", []):
                    mkt_key = mkt.get("key")
                    if mkt_key not in ["h2h", "spreads", "totals"]:
                        continue

                    for outcome in mkt.get("outcomes", []):
                        rows.append({
                            "snapshot_ts": last_update,
                            "game_date": date_obj.isoformat(),
                            "commence_time": commence,
                            "event_id": event_id,
                            "home_team": home,
                            "away_team": away,
                            "book": bk_key,
                            "market_type": mkt_key,
                            "team": outcome.get("name"),
                            "odds": outcome.get("price"),
                            "point": outcome.get("point"),
                        })

        return rows

    except Exception as e:
        print("  REQUEST FAILED:", e)
        return []


# ---------------------------------------
# 4) Main loop (credit optimized)
# ---------------------------------------
for date_obj in dates_needed:

    daily_rows = []

    # US region = DraftKings + FanDuel
    daily_rows.extend(scrape_date_region(date_obj, "us", BOOKS_US))
    time.sleep(1.1)

    # EU region = Pinnacle + Betfair
    daily_rows.extend(scrape_date_region(date_obj, "eu", BOOKS_EU))
    time.sleep(1.1)

    if not daily_rows:
        continue

    daily_df = pd.DataFrame(daily_rows)

    # Append safely to disk
    if os.path.exists(out_path):
        daily_df.to_csv(out_path, mode="a", header=False, index=False)
    else:
        daily_df.to_csv(out_path, mode="w", header=True, index=False)

    print(f"Saved {len(daily_rows)} odds rows for {date_obj}")

print("All done — no wasted credits.")


Total game dates to fetch: 571
[2022-02-16] region=us books=draftkings,fanduel
[2022-02-16] region=eu books=pinnacle,betfair_ex_eu
Saved 146 odds rows for 2022-02-16
[2022-02-17] region=us books=draftkings,fanduel
[2022-02-17] region=eu books=pinnacle,betfair_ex_eu
Saved 194 odds rows for 2022-02-17
[2022-02-18] region=us books=draftkings,fanduel
[2022-02-18] region=eu books=pinnacle,betfair_ex_eu
Saved 140 odds rows for 2022-02-18
[2022-02-19] region=us books=draftkings,fanduel
[2022-02-19] region=eu books=pinnacle,betfair_ex_eu
Saved 136 odds rows for 2022-02-19
[2022-02-20] region=us books=draftkings,fanduel
[2022-02-20] region=eu books=pinnacle,betfair_ex_eu
Saved 144 odds rows for 2022-02-20
[2022-02-21] region=us books=draftkings,fanduel
[2022-02-21] region=eu books=pinnacle,betfair_ex_eu
Saved 74 odds rows for 2022-02-21
[2022-02-22] region=us books=draftkings,fanduel
[2022-02-22] region=eu books=pinnacle,betfair_ex_eu
Saved 146 odds rows for 2022-02-22
[2022-02-23] region=us bo

In [271]:
import pandas as pd
import numpy as np

# -------------------------------------------------
# 1. Load both files
# -------------------------------------------------
odds_old = pd.read_csv("nhl_historical_odds_2020plus_pinnacle_betfair_dk_fd_ml_spreads_totals.csv", low_memory=False)
odds_new = pd.read_csv("odds_2022_02_16_to_present_four_books.csv", low_memory=False)


# -------------------------------------------------
# 2. Standardize COLUMN NAMES
# -------------------------------------------------

# Rename old fields → match new format
odds_old = odds_old.rename(columns={
    "bookmaker": "book",
    "market": "market_type",
    "outcome_name": "team",
    "price": "odds"
})

# Old file has no `game_date`, so compute it from commence_time
odds_old["commence_time"] = pd.to_datetime(odds_old["commence_time"], utc=True, errors="coerce")

odds_old["game_date"] = (
    odds_old["commence_time"]
    .dt.tz_convert("US/Eastern")
    .dt.normalize()
    .dt.tz_localize(None)
)

# New file game_date parsing
odds_new["game_date"] = pd.to_datetime(odds_new["game_date"], errors="coerce")


# -------------------------------------------------
# 3. Force same column order
# -------------------------------------------------

columns_target = [
    "snapshot_ts",
    "game_date",
    "commence_time",
    "event_id",
    "home_team",
    "away_team",
    "book",
    "market_type",
    "team",
    "odds",
    "point"
]

# Add missing columns in either file
for col in columns_target:
    if col not in odds_old.columns:
        odds_old[col] = np.nan
    if col not in odds_new.columns:
        odds_new[col] = np.nan

# Reorder both
odds_old = odds_old[columns_target]
odds_new = odds_new[columns_target]


# -------------------------------------------------
# 4. Combine into one dataset
# -------------------------------------------------

odds = pd.concat([odds_old, odds_new], ignore_index=True)

print("Combined odds shape:", odds.shape)
print("Sample:")
print(odds.head())

# -------------------------------------------------
# 5. Save unified odds file
# -------------------------------------------------

odds.to_csv("odds.csv", index=False)
print("Saved unified odds.csv")


Combined odds shape: (149790, 11)
Sample:
            snapshot_ts  game_date              commence_time  \
0  2020-08-01T23:55:00Z 2020-08-01  2020-08-02 02:30:00+00:00   
1  2020-08-01T23:55:00Z 2020-08-01  2020-08-02 02:30:00+00:00   
2  2020-08-01T23:55:00Z 2020-08-02  2020-08-02 18:00:00+00:00   
3  2020-08-01T23:55:00Z 2020-08-02  2020-08-02 18:00:00+00:00   
4  2020-08-01T23:55:00Z 2020-08-02  2020-08-03 00:00:00+00:00   

                           event_id            home_team  \
0  b2020f52aad491b409755933742b3603       Calgary Flames   
1  b2020f52aad491b409755933742b3603       Calgary Flames   
2  730b76ebabbb9841b784370e52c3506b  Nashville Predators   
3  730b76ebabbb9841b784370e52c3506b  Nashville Predators   
4  d70b6475db17c5bd9ca91bce63651878  Toronto Maple Leafs   

               away_team        book market_type                   team  odds  \
0          Winnipeg Jets  draftkings         h2h         Calgary Flames  -127   
1          Winnipeg Jets  draftkings        

In [275]:
odds.game_date.max()

Timestamp('2024-06-24 00:00:00')